In [1]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_loader import get_yield_curve_history, get_credit_spreads

curve_history = get_yield_curve_history()
spread_history = get_credit_spreads()

print(curve_history.tail())
print(spread_history.tail())

              1M    3M    6M    1Y    2Y    5Y   10Y   30Y
date                                                      
2026-07-02  3.70  3.82  3.98  3.96  4.14  4.23  4.49  4.98
2026-07-06  3.69  3.87  3.98  3.95  4.13  4.21  4.48  4.99
2026-07-07  3.69  3.86  3.99  4.06  4.19  4.27  4.55  5.05
2026-07-08  3.67  3.87  3.99  4.06  4.21  4.31  4.56  5.06
2026-07-09  3.72  3.83  3.96  4.02  4.16  4.27  4.54  5.05
               AAA       A     BBB      HY
date                                      
2026-07-03  0.0037  0.0063  0.0094  0.0274
2026-07-06  0.0037  0.0062  0.0094  0.0272
2026-07-07  0.0039  0.0063  0.0094  0.0267
2026-07-08  0.0038  0.0063  0.0094  0.0270
2026-07-09  0.0039  0.0063  0.0094  0.0270


In [2]:
# Compute Daily Changes (in bps)
rate_changes_bps = curve_history.diff() * 100   # percentage points -> bps
spread_changes_bps = spread_history.diff() * 100

print(rate_changes_bps.describe())
print(spread_changes_bps.describe())

                1M           3M           6M           1Y           2Y  \
count  2879.000000  2879.000000  2879.000000  2879.000000  2879.000000   
mean      0.128517     0.132338     0.133727     0.130948     0.121570   
std       4.747497     2.945685     2.965256     3.987717     5.184888   
min     -71.000000   -23.000000   -36.000000   -60.000000   -57.000000   
25%      -1.000000    -1.000000    -1.000000    -1.000000    -2.000000   
50%       0.000000     0.000000     0.000000     0.000000     0.000000   
75%       1.000000     1.000000     1.000000     1.000000     2.000000   
max     106.000000    34.000000    27.000000    34.000000    34.000000   

                5Y          10Y          30Y  
count  2879.000000  2879.000000  2879.000000  
mean      0.092393     0.084057     0.081973  
std       5.535828     5.307686     5.069930  
min     -32.000000   -30.000000   -31.000000  
25%      -3.000000    -3.000000    -3.000000  
50%       0.000000     0.000000     0.000000  
75% 

In [3]:
# Identify Realistic Shock Sizes
rate_daily_vol = rate_changes_bps.std()
spread_daily_vol = spread_changes_bps.std()

print("Daily rate volatility (bps):\n", rate_daily_vol)
print("Daily spread volatility (bps):\n", spread_daily_vol)

print("Worst historical single-day rate move (bps):\n", rate_changes_bps.min())
print("Worst historical single-day spread widening (bps):\n", spread_changes_bps.max())

Daily rate volatility (bps):
 1M     4.747497
3M     2.945685
6M     2.965256
1Y     3.987717
2Y     5.184888
5Y     5.535828
10Y    5.307686
30Y    5.069930
dtype: float64
Daily spread volatility (bps):
 AAA    0.011319
A      0.012736
BBB    0.015512
HY     0.070481
dtype: float64
Worst historical single-day rate move (bps):
 1M    -71.0
3M    -23.0
6M    -36.0
1Y    -60.0
2Y    -57.0
5Y    -32.0
10Y   -30.0
30Y   -31.0
dtype: float64
Worst historical single-day spread widening (bps):
 AAA    0.06
A      0.08
BBB    0.11
HY     0.59
dtype: float64


In [4]:
# Portfolio
from src.portfolio import Portfolio
from src.data_loader import get_yield_curve_history, get_credit_spreads, get_latest_curve_by_maturity

curve_history = get_yield_curve_history()
risk_free_curve = get_latest_curve_by_maturity(curve_history)
credit_spread = get_credit_spreads()

holdings = [
    {"name": "UST_2Y",  "face_value": 1_000_000, "coupon_rate": 0.04, "maturity": 2,  "rating": "AAA"},
    {"name": "UST_10Y", "face_value": 500_000,   "coupon_rate": 0.045,"maturity": 10, "rating": "AAA"},
    {"name": "Corp_A_5Y",  "face_value": 300_000, "coupon_rate": 0.05, "maturity": 5,  "rating": "A"},
    {"name": "Corp_BBB_7Y","face_value": 400_000, "coupon_rate": 0.06, "maturity": 7,  "rating": "BBB"},
    {"name": "HY_3Y",      "face_value": 200_000, "coupon_rate": 0.08, "maturity": 3,  "rating": "HY"},
]

portfolio = Portfolio(holdings, risk_free_curve, credit_spread)  # <-- this is your "portfolio" object

In [5]:
print(portfolio.market_values()[1])   # should print total portfolio market value
print(portfolio.weighted_duration())  # should print a duration number, e.g. ~5-6

2395761.924142303
4.148729936980922


In [6]:
# Stress Test
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.stress_test import (
    parallel_shock,
    curve_twist,
    credit_widening,
    compare_to_duration_approximation,
)

# --- Test 1: Parallel shock scenarios (mild, moderate, severe) ---
mild = parallel_shock(portfolio, rate_bps=5, spread_bps_by_rating={"AAA": 1, "A": 1, "BBB": 2, "HY": 7})
moderate = parallel_shock(portfolio, rate_bps=50, spread_bps_by_rating={"AAA": 11, "A": 13, "BBB": 15, "HY": 70})
severe = parallel_shock(portfolio, rate_bps=100, spread_bps_by_rating={"AAA": 15, "A": 20, "BBB": 25, "HY": 100})

print("=== Parallel Shock Scenarios ===")
for name, result in [("mild", mild), ("moderate", moderate), ("severe", severe)]:
    print(f"{name:10s} | P&L: {result['pnl']:>12,.2f} | P&L%: {result['pnl_pct']:.4%}")

# Sanity check: larger shocks should produce larger absolute losses
assert abs(moderate["pnl"]) > abs(mild["pnl"]), "Moderate shock should hurt more than mild shock"
assert abs(severe["pnl"]) > abs(moderate["pnl"]), "Severe shock should hurt more than moderate shock"

# --- Test 2: Curve twist (steepening vs flattening) ---
steepening = curve_twist(portfolio, short_end_bps=-25, long_end_bps=25)
flattening = curve_twist(portfolio, short_end_bps=25, long_end_bps=-25)

print("\n=== Curve Twist Scenarios ===")
print(f"Steepening | P&L: {steepening['pnl']:>12,.2f} | P&L%: {steepening['pnl_pct']:.4%}")
print(f"Flattening | P&L: {flattening['pnl']:>12,.2f} | P&L%: {flattening['pnl_pct']:.4%}")

# --- Test 3: Credit widening in isolation ---
credit_stress = credit_widening(portfolio, spread_bps_by_rating={"AAA": 5, "A": 10, "BBB": 20, "HY": 100})
print("\n=== Credit Widening Scenario ===")
print(f"Credit stress | P&L: {credit_stress['pnl']:>12,.2f} | P&L%: {credit_stress['pnl_pct']:.4%}")

# --- Test 4: Duration+convexity approximation vs full repricing ---
gap_mild = compare_to_duration_approximation(portfolio, rate_bps=5)
gap_moderate = compare_to_duration_approximation(portfolio, rate_bps=50)
gap_severe = compare_to_duration_approximation(portfolio, rate_bps=100)

print("\n=== Duration Approximation vs Full Repricing ===")
for name, result in [("mild", gap_mild), ("moderate", gap_moderate), ("severe", gap_severe)]:
    print(f"{name:10s} | Actual P&L%: {result['actual_pnl_pct']:.4%} | "
          f"Approx P&L%: {result['approx_pnl_pct']:.4%} | "
          f"Gap: {result['pnl_gap_pct']:.6%}")

# Sanity check: approximation gap should grow as shock size increases
assert abs(gap_severe["pnl_gap_pct"]) > abs(gap_mild["pnl_gap_pct"]), \
    "Duration approximation error should be larger for bigger shocks (convexity effect)"

print("\nAll Phase 3 stress test checks passed.")

=== Parallel Shock Scenarios ===
mild       | P&L:    -6,512.55 | P&L%: -0.2718%
moderate   | P&L:   -63,702.13 | P&L%: -2.6590%
severe     | P&L:  -117,279.22 | P&L%: -4.8953%

=== Curve Twist Scenarios ===
Steepening | P&L:    16,532.96 | P&L%: 0.6901%
Flattening | P&L:   -16,384.91 | P&L%: -0.6839%

=== Credit Widening Scenario ===
Credit stress | P&L:   -14,214.03 | P&L%: -0.5933%

=== Duration Approximation vs Full Repricing ===
mild       | Actual P&L%: -0.2071% | Approx P&L%: -0.2071% | Gap: -0.000000%
moderate   | Actual P&L%: -2.0405% | Approx P&L%: -2.0400% | Gap: -0.000480%
severe     | Actual P&L%: -4.0152% | Approx P&L%: -4.0114% | Gap: -0.003791%

All Phase 3 stress test checks passed.


In [7]:
# Phase 4: VAR and ES
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
from src.data_loader import get_yield_curve_history, get_credit_spreads
from src.var_es import historical_simulation_pnl, historical_var_es, monte_carlo_pnl, monte_carlo_var_es

# --- Step 1: Rebuild percent-scale historical data ---
# curve_history is already in percent scale (e.g., 4.05), matching raw FRED output
rate_history_pct = get_yield_curve_history()

# get_credit_spreads() was fixed in Phase 2/3 to return decimal scale (e.g., 0.0094)
# multiply back by 100 to restore percent scale (e.g., 0.94), matching the *100 bps conversion
spread_history_pct = get_credit_spreads() * 100

print("Rate history (percent) tail:\n", rate_history_pct.tail())
print("\nSpread history (percent) tail:\n", spread_history_pct.tail())

# --- Step 2: Historical Simulation VaR/ES ---
hist_pnl_df = historical_simulation_pnl(portfolio, rate_history_pct, spread_history_pct, lookback_days=250)
hist_var, hist_es = historical_var_es(hist_pnl_df["pnl"].values, confidence=0.99)

print("\n=== Historical Simulation ===")
print(f"99% VaR: {hist_var:,.2f}")
print(f"99% ES:  {hist_es:,.2f}")

# --- Step 3: Monte Carlo VaR/ES ---
# Calculate daily volatility (in bps, then converted to decimal) for each factor
print(rate_history_pct.columns.tolist())          # e.g., ['1M','3M','6M','1Y','2Y','5Y','10Y','30Y']
print(portfolio.risk_free_curve.index.tolist())    # e.g., [0.083, 0.25, 0.5, 1, 2, 5, 10, 30]
rate_changes_bps = rate_history_pct.diff().dropna() * 100
spread_changes_bps = spread_history_pct.diff().dropna() * 100

rate_vol = rate_changes_bps.std().to_numpy()      # bps scale, e.g. ~4.75, not 0.000475
spread_vol = spread_changes_bps.std().to_numpy()  # bps scale

print("rate_vol sample:", rate_vol[:3])    # should now show ~[4.7, 2.9, 2.9] roughly
print("spread_vol sample:", spread_vol[:3])

mc_pnl_df = monte_carlo_pnl(portfolio, rate_vol, spread_vol, n_simulations=5000)
print(mc_pnl_df["pnl"].describe())         # std should now be in the thousands, not ~0.33

mc_var, mc_es = monte_carlo_var_es(mc_pnl_df["pnl"].values, confidence=0.99)
print("Monte Carlo 99% VaR:", mc_var, "| ES:", mc_es)

print("\n=== Monte Carlo Simulation ===")
print(f"99% VaR: {mc_var:,.2f}")
print(f"99% ES:  {mc_es:,.2f}")

# --- Step 4: Compare against Phase 3 severe stress scenario ---
print("\n=== Comparison to Phase 3 Severe Stress ===")
print(f"Historical 99% VaR: {hist_var:,.2f}")
print(f"Monte Carlo 99% VaR: {mc_var:,.2f}")
print(f"Phase 3 severe stress P&L: {severe['pnl']:,.2f}")

# Sanity checks
assert hist_es <= hist_var, "ES should represent losses worse than or equal to VaR"
assert mc_es <= mc_var, "ES should represent losses worse than or equal to VaR"
assert abs(severe["pnl"]) > abs(hist_var), "Severe stress scenario should generally exceed statistical 99% VaR"

print("\nAll Phase 4 VaR/ES checks passed.")

Rate history (percent) tail:
               1M    3M    6M    1Y    2Y    5Y   10Y   30Y
date                                                      
2026-07-02  3.70  3.82  3.98  3.96  4.14  4.23  4.49  4.98
2026-07-06  3.69  3.87  3.98  3.95  4.13  4.21  4.48  4.99
2026-07-07  3.69  3.86  3.99  4.06  4.19  4.27  4.55  5.05
2026-07-08  3.67  3.87  3.99  4.06  4.21  4.31  4.56  5.06
2026-07-09  3.72  3.83  3.96  4.02  4.16  4.27  4.54  5.05

Spread history (percent) tail:
              AAA     A   BBB    HY
date                              
2026-07-03  0.37  0.63  0.94  2.74
2026-07-06  0.37  0.62  0.94  2.72
2026-07-07  0.39  0.63  0.94  2.67
2026-07-08  0.38  0.63  0.94  2.70
2026-07-09  0.39  0.63  0.94  2.70

=== Historical Simulation ===
99% VaR: -12,126.86
99% ES:  -12,941.22
['1M', '3M', '6M', '1Y', '2Y', '5Y', '10Y', '30Y']
[0.08333333333333333, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0, 30.0]
rate_vol sample: [4.74749664 2.94568502 2.96525602]
spread_vol sample: [1.13194314 1.27359659 1.5

In [8]:
print("rate_vol sample:", rate_vol[:3])
print("spread_vol sample:", spread_vol[:3])
print("shocks sample (first row):", mc_pnl_df.iloc[0])
print(mc_pnl_df["pnl"].describe())

rate_vol sample: [4.74749664 2.94568502 2.96525602]
spread_vol sample: [1.13194314 1.27359659 1.55116098]
shocks sample (first row): base_value       2.395762e+06
shocked_value    2.397985e+06
pnl              2.222922e+03
pnl_pct          9.278561e-04
Name: 0, dtype: float64
count     5000.000000
mean        42.175958
std       3362.774259
min     -12349.515806
25%      -2153.344775
50%          1.506796
75%       2299.263584
max      12528.702964
Name: pnl, dtype: float64


In [9]:
# Correlatio matrix check for MC simulation
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
from src.var_es import monte_carlo_pnl, monte_carlo_var_es

# Build empirical correlation matrix from historical daily changes
all_changes = rate_changes_bps.join(spread_changes_bps, how="inner")  # align on common dates
corr_matrix = all_changes.corr().to_numpy()

print("Correlation matrix shape:", corr_matrix.shape)
print(all_changes.corr().round(2))  # inspect: rates vs spreads, do they move together or inversely?

# Rerun Monte Carlo with the correlation structure
mc_pnl_corr_df = monte_carlo_pnl(
    portfolio, rate_vol, spread_vol,
    n_simulations=5000, correlation=corr_matrix
)

print(mc_pnl_corr_df["pnl"].describe())

mc_var_corr, mc_es_corr = monte_carlo_var_es(mc_pnl_corr_df["pnl"].values, confidence=0.99)
print("Monte Carlo (correlated) 99% VaR:", mc_var_corr, "| ES:", mc_es_corr)

# Compare all three
print("\n=== Full Comparison ===")
print(f"Historical Simulation VaR: {hist_var:,.2f} | ES: {hist_es:,.2f}")
print(f"Monte Carlo (uncorrelated) VaR: {mc_var:,.2f} | ES: {mc_es:,.2f}")
print(f"Monte Carlo (correlated) VaR: {mc_var_corr:,.2f} | ES: {mc_es_corr:,.2f}")

Correlation matrix shape: (12, 12)
       1M    3M    6M    1Y    2Y    5Y   10Y   30Y   AAA     A   BBB    HY
1M   1.00  0.27  0.22  0.15  0.12  0.10  0.09  0.07 -0.06 -0.04 -0.05 -0.06
3M   0.27  1.00  0.55  0.43  0.37  0.31  0.24  0.15  0.00 -0.07 -0.08 -0.11
6M   0.22  0.55  1.00  0.80  0.69  0.62  0.51  0.38 -0.16 -0.23 -0.28 -0.32
1Y   0.15  0.43  0.80  1.00  0.89  0.81  0.68  0.51 -0.17 -0.25 -0.31 -0.38
2Y   0.12  0.37  0.69  0.89  1.00  0.91  0.79  0.61 -0.17 -0.23 -0.29 -0.36
5Y   0.10  0.31  0.62  0.81  0.91  1.00  0.94  0.81 -0.19 -0.23 -0.29 -0.30
10Y  0.09  0.24  0.51  0.68  0.79  0.94  1.00  0.94 -0.19 -0.19 -0.24 -0.21
30Y  0.07  0.15  0.38  0.51  0.61  0.81  0.94  1.00 -0.20 -0.14 -0.18 -0.11
AAA -0.06  0.00 -0.16 -0.17 -0.17 -0.19 -0.19 -0.20  1.00  0.77  0.78  0.58
A   -0.04 -0.07 -0.23 -0.25 -0.23 -0.23 -0.19 -0.14  0.77  1.00  0.90  0.70
BBB -0.05 -0.08 -0.28 -0.31 -0.29 -0.29 -0.24 -0.18  0.78  0.90  1.00  0.77
HY  -0.06 -0.11 -0.32 -0.38 -0.36 -0.30 -0.21 -0.11  

In [10]:
import numpy as np
from src.factor_attribution import compute_factor_exposures, compute_covariance_matrix, factor_risk_contributions

exposures = compute_factor_exposures(portfolio)
cov_matrix = compute_covariance_matrix(rate_changes_bps, spread_changes_bps)

RC_dollar, RC_pct = factor_risk_contributions(exposures, cov_matrix)

factor_names = list(portfolio.risk_free_curve.index) + list(portfolio.credit_spread.columns)
for name, rc, pct in zip(factor_names, RC_dollar, RC_pct):
    print(f"{name:>6}: RC = {rc:>12,.2f} | {pct:.2%} of total risk")

total_rate_risk = sum(RC_pct[:8])
total_spread_risk = sum(RC_pct[8:])
print(f"\nTotal rate risk: {total_rate_risk:.2%}")
print(f"Total spread risk: {total_spread_risk:.2%}")

assert abs(sum(RC_pct) - 1.0) < 1e-6, "Risk contributions should sum to 100%"

0.08333333333333333: RC =        -0.00 | -0.00% of total risk
  0.25: RC =        -0.00 | -0.00% of total risk
   0.5: RC =        -0.00 | -0.00% of total risk
   1.0: RC =        -0.00 | -0.00% of total risk
   2.0: RC = 6,023,837.02 | 21.33% of total risk
   5.0: RC = 8,919,962.33 | 31.58% of total risk
  10.0: RC = 13,357,530.54 | 47.29% of total risk
  30.0: RC =        -0.00 | -0.00% of total risk
   AAA: RC =   155,163.19 | 0.55% of total risk
     A: RC =    15,154.05 | 0.05% of total risk
   BBB: RC =   -73,396.45 | -0.26% of total risk
    HY: RC =  -151,185.97 | -0.54% of total risk

Total rate risk: 100.19%
Total spread risk: -0.19%


Note on factor attribution interpretation: Total spread risk contribution came out near zero (and slightly negative for BBB/HY) due to the strong negative correlation observed between rate and spread factors in the historical sample. This reflects each factor's marginal contribution to total portfolio variance, not the standalone risk of the credit exposure in isolation — the portfolio's large rate risk and the historically negative rate-spread correlation combine such that credit exposures partially offset rate risk in this variance decomposition. This is a known characteristic of correlation-based factor attribution and should be read alongside standalone risk measures (e.g., Phase 2's spread DV01) rather than in isolation.